# Fine-tune with the Databricks fine-tuning API

This notebook submits a `CHAT_COMPLETION` run using the JSONL files produced by `02_stage_data`. Foundation Model Fine-tuning is deprecated and scheduled for removal on August 14, 2026; this notebook is the legacy baseline for the AI Runtime migration example.

The legacy service requires an AWS workspace in `us-east-1` or `us-west-2`, and the workspace must not use S3 access policies.

In [ ]:
%pip install databricks_genai pyyaml

In [ ]:
dbutils.library.restartPython()

In [ ]:
from pathlib import Path

import yaml
from databricks.model_training import foundation_model as fm

CONFIG_FILE = "config.yaml"


def resolve_config_path() -> Path:
    candidates = []
    if "__file__" in globals():
        candidates.append(Path(__file__).resolve().parent / CONFIG_FILE)
    candidates.append(Path.cwd() / CONFIG_FILE)

    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        notebook_path = context.notebookPath().get()
    except Exception:
        pass
    else:
        candidates.append(
            Path("/Workspace")
            / notebook_path.lstrip("/").rsplit("/", 1)[0]
            / CONFIG_FILE
        )

    for candidate in dict.fromkeys(candidates):
        if candidate.is_file():
            return candidate
    searched = ", ".join(str(path) for path in candidates)
    raise FileNotFoundError(f"Could not find {CONFIG_FILE}; searched: {searched}")


config_path = resolve_config_path()
with config_path.open("r", encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

if not isinstance(config, dict):
    raise ValueError(f"Expected a YAML mapping in {config_path}")


def required_mapping(mapping: dict, key: str) -> dict:
    value = mapping.get(key)
    if not isinstance(value, dict):
        raise ValueError(f"Config value {key} must be a mapping")
    return value


def required_string(mapping: dict, key: str) -> str:
    value = str(mapping.get(key, "")).strip()
    if not value:
        raise ValueError(f"Missing required config value: {key}")
    return value


data_config = required_mapping(config, "data")
fine_tuning_config = required_mapping(config, "fine_tuning")

catalog = required_string(data_config, "catalog")
schema = required_string(data_config, "schema")
volume = required_string(data_config, "volume")
output_dir = required_string(data_config, "output_dir").strip("/")
train_file = required_string(data_config, "train_file")
eval_file = required_string(data_config, "eval_file")
if not output_dir:
    raise ValueError("output_dir must not be empty")
if train_file == eval_file:
    raise ValueError("train_file and eval_file must be different")
if not train_file.endswith(".jsonl") or not eval_file.endswith(".jsonl"):
    raise ValueError("train_file and eval_file must use the .jsonl extension")

train_data_path = (
    f"dbfs:/Volumes/{catalog}/{schema}/{volume}/{output_dir}/{train_file}"
)
eval_data_path = (
    f"dbfs:/Volumes/{catalog}/{schema}/{volume}/{output_dir}/{eval_file}"
)

task_type = required_string(fine_tuning_config, "task_type")
if task_type != "CHAT_COMPLETION":
    raise ValueError("The staged messages format requires task_type=CHAT_COMPLETION")

validate_inputs = fine_tuning_config.get("validate_inputs", True)
if not isinstance(validate_inputs, bool):
    raise ValueError("validate_inputs must be a boolean")

create_parameters = {
    "model": required_string(fine_tuning_config, "model"),
    "train_data_path": train_data_path,
    "eval_data_path": eval_data_path,
    "register_to": required_string(fine_tuning_config, "register_to"),
    "task_type": task_type,
    "training_duration": required_string(
        fine_tuning_config, "training_duration"
    ),
    "context_length": required_string(fine_tuning_config, "context_length"),
    "validate_inputs": validate_inputs,
}

learning_rate = fine_tuning_config.get("learning_rate")
if learning_rate is not None:
    if float(learning_rate) <= 0:
        raise ValueError("learning_rate must be greater than zero")
    create_parameters["learning_rate"] = str(learning_rate)

print(f"Using configuration from {config_path}")
print("Fine-tuning parameters:")
for parameter, value in create_parameters.items():
    print(f"  {parameter}: {value}")

## Submit the training run

In [ ]:
run = fm.create(**create_parameters)
print(f"Submitted fine-tuning run: {run}")
run